# Clase: Fundamentos de Señales — Analógica vs Digital (Estilo Feynman)

**Objetivo docente:** Explicar paso a paso **cada decisión** y **cada variable** para que el alumno pueda **interpretar y codificar** las fórmulas.  
Incluye **dos ejemplos separados**:
1. **Analógica (red eléctrica):** $v(t) = A\sin(2\pi f t + \phi)$  
2. **Digital (detección de cambios de fase):** muestreo $x[n]=v(nT_s)$ y decisión 0–5 V con histéresis

**Reglas de graficado:** usar `matplotlib`, una figura por gráfico, sin estilos ni colores específicos.

## Diccionario de variables (qué significan y qué son)

| Variable | Significado |
|---|---|
| `t` | tiempo continuo (s); eje independiente para la señal analógica |
| `n` | índice discreto de muestra (entero 0..N-1) |
| `t_s` | tiempo discreto (s) de cada muestra: $t_s = n / f_s$ |
| `f` | frecuencia (Hz): cuántas "olas" por segundo |
| `Vrms` | valor eficaz (V); ligado a la potencia equivalente de la senoide |
| `A` | amplitud pico (V): $A=\sqrt{2}\cdot Vrms$ |
| $\phi$ | fase inicial (rad): dónde empieza la ola |
| `v(t)` | señal analógica continua: $v(t)=A\sin(2\pi f t + \phi)$ |
| `f_s` | frecuencia de muestreo (Hz) |
| `T_s` | período de muestreo (s): $T_s=1/f_s$ |
| `x[n]` | muestras de la señal: $x[n]=v(n\cdot T_s)$ |
| `h` | histéresis (V): margen ±h para evitar rebotes en comparador |
| `y[n]` | salida digital (V): 0 o 5 según comparador con histéresis |
| `Vlow` | nivel lógico bajo (V) |
| `Vhigh` | nivel lógico alto (V) |
| `noise_std` | desviación estándar del ruido (V) añadido a $x[n]$ (opcional) |

In [ ]:
# Imports y configuración base
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

%matplotlib inline

@dataclass
class Config:
    # --- Ejemplo 1 (ANALÓGICO) ---
    f_analog: float = 60.0        # [Hz]
    Vrms_analog: float = 127.0    # [V RMS]
    phi_analog: float = 0.0       # [rad]
    T1: float = 0.06              # [s]
    fs_cont: float = 20000.0      # [Hz]

    # --- Ejemplo 2 (DIGITAL) ---
    f_dig: float = 60.0           # [Hz]
    Vrms_dig: float = 127.0       # [V RMS]
    phi_dig: float = 0.0          # [rad]
    fs_sample: float = 4000.0     # [Hz]
    T2: float = 0.06              # [s]
    Vlow: float = 0.0             # [V]
    Vhigh: float = 5.0            # [V]
    h: float = 1.0                # [V]
    noise_std: float = 0.0        # [V]

C = Config()

In [ ]:
# Funciones auxiliares
def amp_from_rms(Vrms: float) -> float:
    return np.sqrt(2) * Vrms

def seno(t, A, f, phi):
    return A * np.sin(2*np.pi*f*t + phi)

def muestrear_seno(n, fs, A, f, phi):
    return A * np.sin(2*np.pi*f*(n/fs) + phi)

def comparador_histeresis(x, h, Vlow, Vhigh):
    y = np.zeros_like(x)
    state = Vlow
    for i in range(len(x)):
        if state == Vlow and x[i] >= +h:
            state = Vhigh
        elif state == Vhigh and x[i] <= -h:
            state = Vlow
        y[i] = state
    return y

## Ejemplo 1 — Señal **Analógica** (red eléctrica)

**Pasos:**
1. Elegir $f, Vrms, \phi$.
2. Convertir $Vrms \to A=\sqrt{2}Vrms$.
3. Construir tiempo continuo.
4. Calcular $v(t)$.
5. Graficar y analizar.

In [ ]:
# Ejemplo 1 Analógica
f   = C.f_analog
Vrms= C.Vrms_analog
phi = C.phi_analog

A = amp_from_rms(Vrms)

fs_cont = C.fs_cont
T = C.T1
N = int(fs_cont*T)
t = np.linspace(0, T, N, endpoint=False)
v = seno(t, A, f, phi)

plt.figure(figsize=(10,3))
plt.plot(t, v)
plt.title("Ejemplo 1 — Señal analógica")
plt.xlabel("Tiempo [s]"); plt.ylabel("Voltaje [V]")
plt.grid(True); plt.show()

print(f"Período T0=1/f={1/f*1000:.2f} ms")
print(f"Amplitud pico A={A:.2f} V")

### Checkpoints Analógica
- Cambia $\phi$ a $\pi/2$. ¿Cómo cambia el inicio de la ola?
- Cambia $f$ a 50 Hz. ¿Qué ocurre con el período $T_0$?
- Cambia $Vrms$. ¿Cómo cambia $A$?

## Ejemplo 2 — Señal **Digital**

**Pasos:**
1. Muestreo $x[n] = v(nT_s)$.
2. Opcional: añadir ruido.
3. Comparador con histéresis → $y[n]$.
4. Graficar salida digital.
5. Discutir aliasing e histéresis.

In [ ]:
# Ejemplo 2 Digital
f   = C.f_dig
Vrms= C.Vrms_dig
phi = C.phi_dig
A   = amp_from_rms(Vrms)

fs = C.fs_sample
Ts = 1/fs
T  = C.T2
N  = int(fs*T)
n  = np.arange(N)
t_s= n/fs

x = muestrear_seno(n, fs, A, f, phi)

if C.noise_std>0:
    x += np.random.normal(0, C.noise_std, size=x.shape)

y = comparador_histeresis(x, C.h, C.Vlow, C.Vhigh)

plt.figure(figsize=(10,3))
plt.step(t_s, y, where='post')
plt.title("Ejemplo 2 — Señal digital 0–5 V")
plt.xlabel("Tiempo [s]"); plt.ylabel("Nivel lógico [V]")
plt.grid(True); plt.show()

### Checkpoints Digital
- Baja $f_s$ a 1000 Hz (< 2f). ¿Qué ocurre? (aliasing)
- Añade ruido (`noise_std=2.0`). ¿Por qué la histéresis ayuda?
- Pon $h=0$. ¿Aparecen rebotes espurios?